# mPower Dataset Characterization

Reproducible Phase 1 descriptive results. Run the Slurm pipeline first. This source notebook contains no participant records or executed outputs; the executed version is saved in private output storage. Full metadata census and pilot audio denominators are explicitly separated. Missing diagnosis is never interpreted as healthy control.

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
out = Path(os.environ['MPOWER_OUTPUT_ROOT'])
tables = out / 'eda/tables'
figures = out / 'eda/figures'
discovery = json.loads((tables/'discovery_summary.json').read_text())
scopes = [s for s in ('full','pilot') if (tables/f'{s}_audio_summary.csv').exists()]
scope = scopes[0]
def table(name):
    p = tables / name
    if p.exists():
        try: display(pd.read_csv(p))
        except pd.errors.EmptyDataError: display(Markdown('No eligible observations.'))
    else: display(Markdown('Unavailable: analysis has not produced this output.'))
def figure(name):
    p = figures / (name+'.png')
    if p.exists(): display(Image(filename=str(p)))
    else: display(Markdown('Figure unavailable for current data.'))
def interpretation(text): display(Markdown(text))
interpretation(f"Metadata: **{discovery['recordings']:,} recordings**, **{discovery['participants']:,} participants**. Audio figures below use **{scope}** scope.")

## 1. Data sources and metadata schema

Which files and fields actually exist? Consolidated metadata and download shards are audited separately; shards are not extra observations.

In [ ]:
schemas=json.loads((out/'eda/diagnostics/table_schemas.json').read_text())
display(pd.DataFrame([{k:v for k,v in s.items() if k in ('table','rows','columns','duplicate_rows','error')} for s in schemas]))
display(pd.DataFrame(next(s for s in schemas if s.get('table')=='voice_activity/Voice_Activity.csv')['fields']))
interpretation(f"Inventory: {discovery['files']:,} files; {discovery['total_bytes']:,} bytes. Observed relationships: {discovery['relationships']}")

## 2. Cohort identification and participant demographics

Can age and sex confounding be evaluated? Only explicitly configured audited demographic fields are used.

In [ ]:
table('cohort_characteristics.csv')
table('demographic_coverage.csv')

if discovery.get('demographics_table'):
    table('demographics_by_diagnosis.csv')
    display(json.loads((tables/'demographic_imbalance.json').read_text()))
    figure('demographic_distributions')
    table('age_sex_diagnosis_contingency.csv')
    interpretation('Baseline demographics linked by audited participant key. Age and reported gender can confound associations; unmatched participants remain unknown.')
else:
    interpretation('Age and sex unavailable in voice-only export.')

## 3. Diagnosis distribution and label validity

Does the local export support a binary PD cohort? Unknown and conflicting diagnoses remain distinct.

In [ ]:
figure('diagnosis_availability')
interpretation(f"Participant labels: {discovery['participant_diagnoses']}. Recording labels: {discovery['recording_diagnoses']}. Medication timing alone is not evidence of professional diagnosis; linked source diagnosis is participant-reported.")
table('medication_by_diagnosis.csv')

## 4. Recording frequency and longitudinal characteristics

How much do repeated recordings and prolific contributors affect the data?

In [ ]:
figure('longitudinal_recordings')
interpretation(f"Repeated-recording participants: {discovery['longitudinal_participants']:,}/{discovery['participants']:,}. Top 1% recording share: {discovery['top_1_percent_recording_share']:.2%}.")
display(pd.Series(discovery['recordings_per_participant']))
display(pd.Series(discovery['followup_days']))

## 5. Audio duration and technical characteristics

Are recordings complete, decodable and technically consistent? Duration comparisons describe processed audio only.

In [ ]:
table(f'{scope}_audio_summary.csv')
table('duration_threshold_counts.csv')
figure('audio_duration')
interpretation('No 10-second truncation is applied. A configured resource guard rejects very long files for review. Diagnosis comparisons and baseline-age scatter are exported when demographics are linked.')

## 6. Recording quality assessment

Which technical and acoustic warnings occur? Energy activity and single-channel SNR are estimates, not ground truth.

In [ ]:
figure('qc_distributions')
table('qc_threshold_sensitivity.csv')
interpretation('Low intensity and irregular voicing may reflect dysphonia; they are not grounds for automatic primary exclusion. Diagnosis-specific QC counts are exported when demographics are linked; inspect pilot denominators before interpreting differences.')
for name in ('decoded','flag_short','flag_near_silent','flag_clipping','flag_low_estimated_snr'):
    figure('example_'+name)

## 7. Acoustic feature extraction methodology

Can established algorithms extract interpretable measurements reproducibly?

In [ ]:
display(pd.read_csv(out/'processed/feature_dictionary.csv'))
table('praat_opensmile_comparison.csv')
interpretation('Praat full and contiguous-pitched intervals, openSMILE eGeMAPSv02 and librosa features have separate namespaces. Undefined perturbation measures remain missing. CPP and formants are not validated. Pitch and trim sensitivity outputs are in the private feature table.')

## 8. Acoustic feature distributions and correlations

What are the observed feature ranges and validity rates?

In [ ]:
table(f'{scope}_feature_summary.csv')
figure('acoustic_distributions')
figure('selected_feature_correlations')
interpretation('These are descriptive pilot distributions when scope=pilot; no population or PD effect is inferred from them. Participant-median Spearman matrices are saved separately.')

## 9. PD versus control exploratory comparisons

Do acoustic features differ by diagnosis after accounting for repeated observations?

In [ ]:
table('exploratory_pd_associations.csv')

if discovery.get('demographics_table'):
    table('covariate_adjusted_pd_associations.csv')
    table('quality_by_diagnosis.csv')
    figure('diagnosis_duration_frequency')
    interpretation('Participant-median rank comparisons and age/gender/quality-adjusted HC3 OLS are exploratory. BH FDR is reported; pilot selection limits generalization. No causal biomarker claim.')
else:
    interpretation('No PD associations without verified labels.')

## 10. Duplicate/leakage audit

Can source or audio duplicates contaminate partitions?

In [ ]:
display(json.loads((out/'eda/diagnostics/duplicate_summary.json').read_text()))
interpretation(f"Duplicate source recording-ID rows: {discovery['duplicate_recording_ids']}. Audio hash audit scope: {scope}. Near-duplicate analysis remains unresolved. Detected duplicate-connected participants cannot cross splits.")

## 11. Quality filtering and cohort selection

What remains under each eligibility or QC rule?

In [ ]:
table('qc_exclusion_flow.csv')
display(json.loads((tables/'cohort_readiness.json').read_text()))
table('age_gender_matching_flow.csv')
interpretation('Unknown and conflicting labels remain outside binary cohorts. Cohort C matches participant counts within reported gender and 5-year age bins between ages 50–80; review residual age imbalance. Processed-scope cohorts remain provisional.')

## 12. Final participant-level splits and dataset readiness

Are independent training, validation and test partitions available?

In [ ]:
display(json.loads((out/'eda/diagnostics/split_status.json').read_text()))
for cohort in ('A','B','C'): table(f'cohort_{cohort}_splits.csv')
interpretation('No learned transformations or foundation-model embeddings were fitted. Reuse fixed participant assignments across sensitivity cohorts; do not claim supervised readiness until label and leakage audits pass.')

## 13. Limitations, confounders, and recommended next steps

Which conclusions are supported, and which analyses remain blocked?

In [ ]:
display(Markdown((out/'eda/MPOWER_FINAL_REPORT.md').read_text()))